# Tono · la pregunta central, por fin medible

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

El proyecto existe para comprobar si un modelo de cáncer de piel funciona igual
de bien en piel oscura. Hasta ahora no se podía medir: PAD-UFES-20 tiene **6**
casos malignos en tonos 4-6, y el portal de Stanford que sirve DDI está caído.

**Fitzpatrick17k lo desbloquea:**

| Dataset | Imágenes en tonos 4-6 | Malignas |
|---|---|---|
| PAD-UFES-20 | 6 | 6 |
| Fitzpatrick17k | **1.079** | **509** |

## Dos experimentos

**A — Transferencia.** Un modelo entrenado con PAD-UFES (población brasileña,
sesgada a piel clara, fotos de móvil) se evalúa sobre Fitzpatrick17k
**estratificando por tono**. Es la pregunta directa: ¿falla más en piel oscura
un modelo entrenado sobre piel clara?

**B — Entrenamiento con representación.** Se entrena sobre Fitzpatrick17k, que
sí tiene piel oscura, y se audita igual. Si la brecha desaparece, la causa era
la composición de los datos, no la tarea.

## El tono va en bandas, no en seis clases

Las dos anotaciones de Fitzpatrick del dataset coinciden exactamente solo el
**47,9%** de las veces (±1 tono: 91%). Seis clases exactas medirían sobre todo
el ruido del anotador, así que se agrupa en clara (1-2), media (3-4) y oscura
(5-6).

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', '/tmp/' + nombre], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/tmp/' + nombre], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert 'sm_%d%d' % cap in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')
print('entradas:', os.listdir('/kaggle/input'))

## 1. Manifiestos

In [ ]:
OUT = '/kaggle/working'
os.chdir('/tmp/tono')

anclas = glob.glob('/kaggle/input/**/fitzpatrick17k*.csv', recursive=True)
FITZ = os.path.dirname(anclas[0]) if anclas else None
print('Fitzpatrick17k en:', FITZ)

pad = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)
       if 'fitzpatrick' not in os.path.basename(p).lower()]
PAD = os.path.dirname(pad[0]) if pad else None
while PAD and PAD != '/kaggle/input' and not glob.glob(os.path.join(PAD, '**', '*.png'), recursive=True):
    PAD = os.path.dirname(PAD)
print('PAD-UFES en:', PAD)

In [ ]:
!python -m datos.fitzpatrick17k --root {FITZ} --out {OUT}/manifiesto_fitz.csv
!python -m datos.pad_ufes --root {PAD} --out {OUT}/manifiesto_pad.csv

## 2. Experimento A — ¿transfiere a piel oscura?

Se entrena con PAD-UFES y se evalúa sobre Fitzpatrick17k entero, estratificando
por banda de tono.

In [ ]:
os.chdir('/tmp/cxr')
!python -m src.train --config /tmp/tono/configs/pad_ufes.yaml --manifest {OUT}/manifiesto_pad.csv --out-dir {OUT}/runs/pad
!python -m src.evaluate --checkpoint {OUT}/runs/pad/best.pth --manifest {OUT}/manifiesto_pad.csv --split test --out-dir {OUT}/reports/A_pad_interno --n-boot 1000
!python -m src.evaluate --checkpoint {OUT}/runs/pad/best.pth --manifest {OUT}/manifiesto_fitz.csv --split all --out-dir {OUT}/reports/A_pad_en_fitz --n-boot 1000

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/A_pad_en_fitz/predictions.csv --out-dir {OUT}/reports/A_equidad --attributes fitzpatrick,fitzpatrick_exacto --intersect "" --min-n 40

## 3. Experimento B — entrenar con representación

Mismo backbone, misma configuración salvo el dataset.

In [ ]:
!python -m src.train --config /tmp/tono/configs/fitzpatrick17k.yaml --manifest {OUT}/manifiesto_fitz.csv --out-dir {OUT}/runs/fitz
!python -m src.evaluate --checkpoint {OUT}/runs/fitz/best.pth --manifest {OUT}/manifiesto_fitz.csv --split test --out-dir {OUT}/reports/B_fitz_interno --n-boot 1000
!python -m src.fairness --predictions {OUT}/reports/B_fitz_interno/predictions.csv --out-dir {OUT}/reports/B_equidad --attributes fitzpatrick,fitzpatrick_exacto --intersect "" --min-n 30

## 4. Comparación

In [ ]:
import pandas as pd

def por_banda(ruta, titulo):
    if not os.path.exists(ruta):
        print(titulo, ': no disponible'); return None
    t = pd.read_csv(ruta)
    t = t[t.atributo == 'fitzpatrick'].sort_values('subgrupo')
    print()
    print('===', titulo, '===')
    print(t[['subgrupo', 'n', 'prevalencia', 'auroc', 'sensibilidad',
             'fnr_infradiagnostico']].to_string(index=False))
    return t

a = por_banda(OUT + '/reports/A_equidad/subgrupos.csv', 'A: entrenado en PAD-UFES (piel clara)')
b = por_banda(OUT + '/reports/B_equidad/subgrupos.csv', 'B: entrenado en Fitzpatrick17k')

for etiqueta, t in [('A', a), ('B', b)]:
    if t is not None and len(t) >= 2:
        brecha = float(t.fnr_infradiagnostico.max() - t.fnr_infradiagnostico.min())
        peor = t.loc[t.fnr_infradiagnostico.idxmax(), 'subgrupo']
        print()
        print('%s: brecha de FNR entre bandas = %.4f  (peor: %s)' % (etiqueta, brecha, peor))

## 5. Resumen

In [ ]:
resumen = {'pregunta': 'funciona igual de bien en piel oscura?',
           'nota_tono': 'bandas 1-2 / 3-4 / 5-6; las dos anotaciones del dataset coinciden exacto solo 47.9%',
           'salvedad_etiqueta': 'Fitzpatrick17k son atlas dermatologicos: diagnostico clinico, no biopsia'}

for nombre, ruta in [('A_pad_interno', OUT + '/reports/A_pad_interno/metrics.json'),
                     ('A_pad_en_fitz', OUT + '/reports/A_pad_en_fitz/metrics.json'),
                     ('B_fitz_interno', OUT + '/reports/B_fitz_interno/metrics.json'),
                     ('A_equidad', OUT + '/reports/A_equidad/fairness.json'),
                     ('B_equidad', OUT + '/reports/B_equidad/fairness.json')]:
    if os.path.exists(ruta):
        d = json.load(open(ruta))
        resumen[nombre] = {k: v for k, v in d.items() if k != 'detalle'}

if a is not None:
    resumen['A_por_banda'] = a[['subgrupo', 'n', 'auroc', 'fnr_infradiagnostico']].to_dict('records')
if b is not None:
    resumen['B_por_banda'] = b[['subgrupo', 'n', 'auroc', 'fnr_infradiagnostico']].to_dict('records')

resumen['minutos'] = round((time.time() - T0) / 60, 1)
json.dump(resumen, open(OUT + '/resumen.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False)[:3000])

import shutil
for f_ in glob.glob(OUT + '/manifiesto_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))